# Sweetviz Lesson: Olist Payments Edition

A hands-on guide to the best uses of `sweetviz` for real-world EDA, using `olist_order_payments_dataset.csv`.

**Contents**
1. Core value and when to use Sweetviz
2. Setup
3. Example 1: Clean single-dataset profile
4. Example 2: Target analysis on `payment_value`
5. Example 3: Intra-dataset comparison (credit card vs. other)
6. Saving reports as standalone HTML
7. Cheat sheet

## 1. Core value and when to use it

Sweetviz is a **first-five-minutes** tool. In two lines you get distributions, missing values, duplicates, associations, target breakdowns, and population comparisons for every column. Writing that by hand in pandas/Seaborn takes 30+ lines per table.

| Use Sweetviz for | Use pandas / Seaborn / DuckDB for |
|---|---|
| First look at a new CSV | Final, polished charts for a write-up |
| "What differs between group A and group B?" | Custom aggregates, joins, window logic |
| Spotting skew, zeros, odd categories, missing data | Anything that must be reproducible in a pipeline |
| Quick target-vs-feature screening | Time-series work |

### Pitfalls to avoid
- **Skip ID columns** (`order_id`, `customer_id`, `product_id`). They are ~100% unique, so they add clutter and slow the associations.
- **Force discrete numbers to categorical.** `payment_installments` and `payment_sequential` are integers but behave like categories. Without `force_cat` you get a histogram instead of per-value breakdowns.
- **Timestamps in the `orders` file load as strings.** Convert them and extract month or weekday, or skip them.
- **Sample huge tables.** `olist_geolocation` has about 1M rows; use `df.sample(100_000, random_state=42)`.
- **Sweetviz only accepts pandas DataFrames.** If you aggregate in DuckDB, finish with `.df()`.
- **`show_notebook` embeds the whole report in the `.ipynb` output.** Clear outputs before committing, or the notebook file balloons.

### Install check
Sweetviz is barely maintained and Python 3.14 is very new, so 3.14 support is unconfirmed. Run the cell below first. If you hit numpy-2 related errors, a community fork (`sweetviz-cb`) was created to fix numpy 2.0+ compatibility; install it with `pip install sweetviz-cb` and keep using `import sweetviz as sv`. If that also fails, a throwaway Python 3.12/3.13 venv for this notebook is the fallback.

In [ ]:
# Install check: this should import without errors
import sweetviz as sv
print("sweetviz imported OK, version:", getattr(sv, "__version__", "unknown"))

## 2. Setup

This cell searches for the payments CSV under `C:\python-data-analytics\olist\`, so it works wherever the notebook itself is saved. It also silences the harmless `findfont: Failed to find font weight normal for Roboto` warnings that Sweetviz triggers on Windows (matplotlib just falls back to another font, so only chart text styling is affected).

The payments file has five columns: `order_id`, `payment_sequential`, `payment_type`, `payment_installments`, `payment_value`.

In [ ]:
import logging
import pandas as pd
import sweetviz as sv
from pathlib import Path

# Silence harmless matplotlib "findfont: Failed to find font ... Roboto" warnings
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

BASE = Path(r"C:\python-data-analytics")

# Find the Olist CSVs wherever they live under the "olist" folder
matches = list((BASE / "olist").rglob("olist_order_payments_dataset.csv"))
assert matches, f"CSV not found under {BASE / 'olist'} - check BASE"
csv_path = matches[0]

DATA = csv_path.parent
REPORTS = BASE / "reports"
REPORTS.mkdir(exist_ok=True)

print("Data folder:", DATA)
pay = pd.read_csv(csv_path)
print(pay.shape)
pay.head()

## 3. Example 1: Clean single-dataset profile

`FeatureConfig` does two jobs here: it **skips** the high-cardinality ID and **forces** the discrete integer columns to categorical.

In [ ]:
cfg = sv.FeatureConfig(
    skip="order_id",
    force_cat=["payment_sequential", "payment_installments"],
)

rep_profile = sv.analyze([pay, "Order Payments"], feat_cfg=cfg)
rep_profile.show_notebook(w="100%", h="900", scale=0.85)

**What to check in the report** (verify against your own output):
- `payment_type` is dominated by `credit_card`, with a tiny `not_defined` category.
- `payment_value` is heavily right-skewed, with a few zero-value rows.
- `payment_installments` has a handful of `0` values, which are suspicious.
- `payment_sequential` above 1 means the order was split across several payments.

## 4. Example 2: Target analysis on `payment_value`

For each feature, Sweetviz overlays the **average target value per category** on the distribution, so you can see how mean `payment_value` moves across installment counts and payment types.

In [ ]:
rep_target = sv.analyze(
    [pay, "Order Payments"],
    target_feat="payment_value",
    feat_cfg=cfg,
)
rep_target.show_notebook(w="100%", h="900", scale=0.85)

**Two caveats**
- **Direction of causality.** People often choose more installments *because* the purchase is large. Installments correlate with value, but they don't necessarily drive it.
- **Skew.** Means are dominated by outliers. Confirm anything interesting with medians (cell below).

In [ ]:
# Sanity-check the mean-based view with medians
(
    pay.groupby("payment_installments")["payment_value"]
       .agg(["count", "median", "mean"])
       .round(2)
       .head(15)
)

## 5. Example 3: Intra-dataset comparison (credit card vs. other)

`compare_intra` splits one DataFrame by a boolean Series and compares the two halves side by side. We **skip `payment_type`** because it defines the split and would be trivially different.

> **Grain warning:** each row in this file is a **payment**, not an order. One order can mix a credit card and a voucher, so "credit card orders" here really means "credit-card payment rows". For a true order-level comparison, aggregate to one row per `order_id` first (DuckDB `GROUP BY`, then `.df()`), then compare.

In [ ]:
is_cc = pay["payment_type"] == "credit_card"

cfg_cc = sv.FeatureConfig(
    skip=["order_id", "payment_type"],
    force_cat=["payment_sequential", "payment_installments"],
)

rep_cc = sv.compare_intra(
    pay,
    is_cc,
    ["Credit Card", "Non-Credit Card"],
    target_feat="payment_value",
    feat_cfg=cfg_cc,
)
rep_cc.show_notebook(w="100%", h="900", scale=0.85)

## 6. Saving reports as standalone HTML

Saved HTML files open in any browser with no Python needed. If the inline view ever renders blank in VS Code, open the saved file in a browser instead. Consider adding `reports/` to `.gitignore`, since these files are regenerable.

In [ ]:
reports = {
    "sweetviz_payments_report.html": rep_profile,
    "sweetviz_payments_target_value.html": rep_target,
    "sweetviz_payments_cc_vs_other.html": rep_cc,
}

for fname, rep in reports.items():
    rep.show_html(
        filepath=str(REPORTS / fname),
        open_browser=False,
        layout="widescreen",
        scale=None,
    )
    print("saved:", REPORTS / fname)

**Display tips:** if the report feels cramped in a narrow VS Code pane, try `scale=0.7` or `layout="vertical"` in `show_notebook`.

## 7. Cheat sheet

| Function / parameter | What it does | Remember |
|---|---|---|
| `sv.analyze([df, "Name"], target_feat=, feat_cfg=)` | Profile one DataFrame, optionally against a target | Target must be numeric or boolean |
| `sv.compare_intra(df, mask, ["A", "B"], ...)` | Split one DataFrame by a boolean Series and compare the halves | Skip the column that defines the split |
| `sv.compare([df1, "A"], [df2, "B"], ...)` | Compare two separate DataFrames (train/test, period vs period) | Same column names needed in both |
| `sv.FeatureConfig(skip=, force_cat=, force_num=, force_text=)` | Drop columns and override type detection | Skip IDs; `force_cat` for discrete integers |
| `.show_notebook(w, h, scale, layout)` / `.show_html(filepath, open_browser, layout, scale)` | Inline display vs. saved HTML file | Also `pairwise_analysis="off"` in `analyze` to speed up wide tables |

### Next step
Run all three reports, then write down **three questions** the reports raise but can't answer. Those become your pandas/DuckDB investigations, which is where Sweetviz hands off to real analysis.